In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import math
import random

import tqdm.notebook
from tqdm.notebook import trange

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
class ResNet(nn.Module):
    def __init__(self, game, num_resblocks, num_hidden, device):
        super().__init__()
        self.device = device # remembered so MCTS / training can send inputs to the same place

        self.startBlock = nn.Sequential(
            nn.Conv2d(3, num_hidden, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_hidden),
            nn.ReLU(),
        )

        self.backbone = nn.ModuleList(
            [ResBlock(num_hidden) for _ in range(num_resblocks)]
        )

        self.policy_head = nn.Sequential(
            nn.Conv2d(num_hidden, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32* game.row_ct * game.col_ct, game.action_sz)
        )

        self.value_head = nn.Sequential(
            nn.Conv2d(num_hidden, 3, kernel_size=3, padding=1),
            nn.BatchNorm2d(3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(3 * game.row_ct * game.col_ct , 1),
            nn.Tanh()
        )

        self.to(device) # move all weights to the GPU (or CPU)

    def forward(self, x):
        x = self.startBlock(x)
        for resBlock in self.backbone:
            x = resBlock(x)

        policy = self.policy_head(x)
        value = self.value_head(x)
        return policy, value



class ResBlock(nn.Module):
    def __init__(self, num_hidden):
        super().__init__()
        self.conv1 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(num_hidden)
        self.conv2 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_hidden)

    def forward(self, x):
        residual = x    
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = x + residual
        x = F.relu(x)
        return x